In [1]:
import matplotlib.pyplot as plt
from utils.dataset import FreiHand2DDataset, heatmaps_to_coords, generate_heatmaps
from utils.fh_utils import plot_hand, sample_version
import numpy as np
from training.model import UNetKeypoint, UNetKeypointLight
import torch.nn as nn
from torch.utils.data import random_split, DataLoader
import wandb
import torch.nn.functional as F
import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'utils.dataset'

# Training

In [3]:
data_percent = 0.5
dataset = FreiHand2DDataset("data", version=sample_version.sample)
dataset, _ = random_split(dataset, [int(len(dataset) * data_percent), len(dataset) - int(len(dataset) * data_percent)])

val_percent = 0.1
val_size = int(len(dataset) * val_percent)
train_size = len(dataset) - val_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

Loading FreiHAND dataset index ...


Loading of 32560 samples done in 2.43 seconds


In [4]:
def visualize_joints(img, joints_gt, joints_pred):
    """Generate GT vs predicted joint overlay image."""
    img = img.permute(1, 2, 0).cpu().numpy()
    fig, ax = plt.subplots()
    ax.imshow(img)
    ax.scatter(joints_gt[:, 0], joints_gt[:, 1], c='g', label='GT', s=15)
    ax.scatter(joints_pred[:, 0], joints_pred[:, 1], c='r', label='Pred', s=15)
    ax.axis('off')
    ax.legend()
    return fig

In [5]:
def train(model, train_loader, val_loader, optimizer, device, epochs=10):
    model.to(device)
    global_step = 0  # Track total steps across epochs

    for epoch in range(epochs):
        model.train()
        train_loss = 0

        for batch in train_loader:
            images = batch['image'].to(device)            # [B, 3, H, W]
            joints = batch['joints_uv']                   # [B, 21, 2]

            gt_heatmaps = generate_heatmaps(joints)       # [B, 21, H, W]
            gt_heatmaps = gt_heatmaps.to(device)

            pred_heatmaps = model(images)                 # [B, 21, H, W]
            loss = nn.MSELoss()(pred_heatmaps, gt_heatmaps)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            global_step += 1  # Increment step counter


            wandb.log({
                "train_loss": loss.item(),
                "epoch": epoch + 1,
            })

            # 🔍 Validation + Logging images
            if global_step % 10 == 0:
                model.eval()
                with torch.no_grad():
                    val_loss = 0
                    for i, batch in enumerate(val_loader):
                        images = batch['image'].to(device)
                        joints = batch['joints_uv']
                        gt_heatmaps = generate_heatmaps(joints).to(device)
                        pred_heatmaps = model(images)
                        loss = F.mse_loss(pred_heatmaps, gt_heatmaps)
                        val_loss += loss.item()

                        wandb.log({
                            "val_loss": loss.item(),
                            "global_step": global_step,
                        })
                        print(f"[Step {global_step}] Val loss: {val_loss / len(val_loader):.4f}")

                        if i == 0:  # log only first val batch
                            pred_coords = heatmaps_to_coords(pred_heatmaps)
                            images_np = images.cpu()
                            gt_coords = joints.cpu()
                            pred_coords = pred_coords.cpu()

                            log_images = {}

                            for j in range(min(4, images_np.shape[0])):
                                fig = visualize_joints(images_np[j], gt_coords[j], pred_coords[j])
                                fig.savefig(f"output_{j}.png")
                                log_images[f"val/sample_{j}"] = wandb.Image(fig)
                                plt.close(fig)
                            wandb.log(log_images, step=global_step)
                            print(f"Logged image for step {global_step}")  # Confirm this prints

                wandb.log({"val_loss": val_loss / len(val_loader)})
                print(f"[Epoch {epoch+1}] Validation done. Avg val loss: {val_loss / len(val_loader):.4f}")

In [6]:
wandb.init(project="freihand-heatmap-2", reinit=True)
model = UNetKeypointLight(num_joints=21)  # your architecture
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
train(model, train_loader, val_loader, optimizer, device='cuda', epochs=1)

wandb: Currently logged in as: bezpalaya-katya (nn24) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


[Step 10] Val loss: 0.0008
Logged image for step 10
[Step 10] Val loss: 0.0017
[Step 10] Val loss: 0.0025
[Step 10] Val loss: 0.0034
[Step 10] Val loss: 0.0042
[Step 10] Val loss: 0.0051
[Step 10] Val loss: 0.0059
[Step 10] Val loss: 0.0068
[Step 10] Val loss: 0.0076
[Step 10] Val loss: 0.0085


wandb: WARNING Tried to log to step 10 that is less than the current step 11. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.


[Step 10] Val loss: 0.0093
[Step 10] Val loss: 0.0102
[Step 10] Val loss: 0.0110
[Step 10] Val loss: 0.0119
[Step 10] Val loss: 0.0127
[Step 10] Val loss: 0.0136
[Step 10] Val loss: 0.0144
[Step 10] Val loss: 0.0153
[Step 10] Val loss: 0.0161
[Step 10] Val loss: 0.0170
[Step 10] Val loss: 0.0178
[Step 10] Val loss: 0.0187
[Step 10] Val loss: 0.0195
[Step 10] Val loss: 0.0204
[Step 10] Val loss: 0.0212
[Step 10] Val loss: 0.0221
[Epoch 1] Validation done. Avg val loss: 0.0221
[Step 20] Val loss: 0.0008
Logged image for step 20
[Step 20] Val loss: 0.0016
[Step 20] Val loss: 0.0024
[Step 20] Val loss: 0.0031
[Step 20] Val loss: 0.0039
[Step 20] Val loss: 0.0047
[Step 20] Val loss: 0.0055
[Step 20] Val loss: 0.0063
[Step 20] Val loss: 0.0071
[Step 20] Val loss: 0.0078
[Step 20] Val loss: 0.0086
[Step 20] Val loss: 0.0094
[Step 20] Val loss: 0.0102
[Step 20] Val loss: 0.0110


wandb: WARNING Tried to log to step 20 that is less than the current step 48. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.


...

wandb: WARNING Tried to log to step 60 that is less than the current step 196. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.


[Step 60] Val loss: 0.0057
[Step 60] Val loss: 0.0063
[Step 60] Val loss: 0.0069
[Step 60] Val loss: 0.0075
[Step 60] Val loss: 0.0080
[Step 60] Val loss: 0.0086
[Step 60] Val loss: 0.0092
[Step 60] Val loss: 0.0098
[Step 60] Val loss: 0.0103
[Step 60] Val loss: 0.0109
[Step 60] Val loss: 0.0115
[Step 60] Val loss: 0.0121
[Step 60] Val loss: 0.0126
[Step 60] Val loss: 0.0132
[Step 60] Val loss: 0.0138
[Step 60] Val loss: 0.0144
[Step 60] Val loss: 0.0149
[Epoch 1] Validation done. Avg val loss: 0.0149
[Step 70] Val loss: 0.0005
Logged image for step 70


wandb: WARNING Tried to log to step 70 that is less than the current step 233. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.


[Step 70] Val loss: 0.0011
[Step 70] Val loss: 0.0016
[Step 70] Val loss: 0.0022
[Step 70] Val loss: 0.0027
[Step 70] Val loss: 0.0033
[Step 70] Val loss: 0.0038
[Step 70] Val loss: 0.0044
[Step 70] Val loss: 0.0049
[Step 70] Val loss: 0.0055
[Step 70] Val loss: 0.0060
[Step 70] Val loss: 0.0066
[Step 70] Val loss: 0.0071
[Step 70] Val loss: 0.0077
[Step 70] Val loss: 0.0082
[Step 70] Val loss: 0.0088
[Step 70] Val loss: 0.0093


KeyboardInterrupt: 

In [7]:
pytorch_total_params = sum(p.numel() for p in model.parameters())

In [8]:
pytorch_total_params

31044821